# Batch Normalization
- 출력값의 분포를 안정화   
- Vanishing gradient 문제가 감소
- 가중치 초기화에 덜 민감  
- 하이퍼 파라미터 탐색을 쉽게 (Normalization을 통해서 학습의 속도가 빨라짐 - learning rate를 높게 할 수 있음)   
- 정규화 효과가 있어서 dropout이 필요 없음(단, 실제로는 그 효과가 미비해서 같이 사용을 많이 함)   
- 신경망과 하이퍼 파라미터의 상관관계를 낮추어 많은 파라미터가 잘 작동하도록 함   
- 오버피팅을 억제   

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if device == 'cuda':
    torch.cuda.manual_seed_all(42)
print(device + " is available")

cuda is available


In [3]:
train_dataset = datasets.FashionMNIST(root='./data',
                                      train=True,
                                      transform=transforms.ToTensor(),
                                      download=True)
test_dataset = datasets.FashionMNIST(root='./data',
                                     train=False,
                                     transform=transforms.ToTensor())

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 212kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.9MB/s]


In [4]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=64,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=64,
                                          shuffle=False)

In [5]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.layer1 = nn.Sequential(
        nn.Linear(784, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True)
    )
    self.layer2 = nn.Sequential(
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True)
    )
    self.layer3 = nn.Sequential(
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(inplace=True)
    )
    self.layer4 = nn.Sequential(
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(inplace=True)
    )
    self.dropout = nn.Dropout(p=0.3)
    self.fc5 = nn.Linear(64, 10)

  def forward(self, x):
    x = x.view(-1, 784)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.dropout(x)
    x = self.fc5(x)
    return x

In [6]:
model=Net().to(device)
criterion=nn.CrossEntropyLoss().to(device)
optimizer=optim.Adam(model.parameters(), lr=0.01)

In [7]:
from torchsummary import summary

summary(model, input_size=(1, 28, 28))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 512]         401,920
       BatchNorm1d-2                  [-1, 512]           1,024
              ReLU-3                  [-1, 512]               0
            Linear-4                  [-1, 256]         131,328
       BatchNorm1d-5                  [-1, 256]             512
              ReLU-6                  [-1, 256]               0
            Linear-7                  [-1, 128]          32,896
       BatchNorm1d-8                  [-1, 128]             256
              ReLU-9                  [-1, 128]               0
           Linear-10                   [-1, 64]           8,256
      BatchNorm1d-11                   [-1, 64]             128
             ReLU-12                   [-1, 64]               0
          Dropout-13                   [-1, 64]               0
           Linear-14                   

In [8]:
num_epochs=10
for epoch in range(num_epochs):
  for idx, (x, y) in enumerate(train_loader):
    x=x.to(device)
    y=y.to(device)

    optimizer.zero_grad()
    h=model(x)
    loss=criterion(h, y)
    loss.backward()
    optimizer.step()

    if idx%100==0:
      print(f'Epoch[{epoch+1}/{num_epochs}] Batch[{idx},{len(train_loader)}] Loss[{loss.item()}]')


Epoch[1/10] Batch[0,938] Loss[2.438943862915039]
Epoch[1/10] Batch[100,938] Loss[0.9714941382408142]
Epoch[1/10] Batch[200,938] Loss[0.4688881039619446]
Epoch[1/10] Batch[300,938] Loss[0.5312017202377319]
Epoch[1/10] Batch[400,938] Loss[0.49910968542099]
Epoch[1/10] Batch[500,938] Loss[0.47108978033065796]
Epoch[1/10] Batch[600,938] Loss[0.37798213958740234]
Epoch[1/10] Batch[700,938] Loss[0.5000079870223999]
Epoch[1/10] Batch[800,938] Loss[0.35174107551574707]
Epoch[1/10] Batch[900,938] Loss[0.3564257025718689]
Epoch[2/10] Batch[0,938] Loss[0.37152397632598877]
Epoch[2/10] Batch[100,938] Loss[0.475504070520401]
Epoch[2/10] Batch[200,938] Loss[0.37434178590774536]
Epoch[2/10] Batch[300,938] Loss[0.6693535447120667]
Epoch[2/10] Batch[400,938] Loss[0.2600126564502716]
Epoch[2/10] Batch[500,938] Loss[0.5120771527290344]
Epoch[2/10] Batch[600,938] Loss[0.34492337703704834]
Epoch[2/10] Batch[700,938] Loss[0.35258397459983826]
Epoch[2/10] Batch[800,938] Loss[0.36650484800338745]
Epoch[2/10] 

In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = correct / total

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Test Accuracy: 0.8847
Test Accuracy: 88.47%
